# Welcome to the Titanic Competition
Let's create our first kaggle notebook and participate to the Titanic Competition: https://www.kaggle.com/competitions/titanic
using the **Logistic Regression**

In [ ]:
# Let's import all librairies needed
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import log_loss, accuracy_score, classification_report
import warnings
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
import re

# 01: Exploratory Data Analysis and Features creation
First, let's dive into the data, change them into numerical values, plot some of them and create our own features

In [ ]:
# Let's import the data 
unprocessed_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
unprocessed_data.head()

In [ ]:
# In order to use the logistic regression method, we'll need numerical values as inputs
# Let's change the sex category as 1: female and 0: male
unprocessed_data['Sex'] = unprocessed_data['Sex'].map({'male': 0, 'female': 1})

In [ ]:
# Let's plot some of our inputs: Age, Sex, and Ticket Class 
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
sns.histplot(
    data=unprocessed_data[['Survived', 'Age']].dropna(),
    x="Age",
    hue="Survived",
    multiple="dodge",
    bins=8,
    shrink=0.8,
    ax=ax1
)
ax1.set_xlabel("Age")
ax1.set_ylabel("Number of passengers")
ax1.set_title("Distribution of survivors by age")

sns.histplot(
    data=unprocessed_data[['Survived', 'Pclass']].dropna(),
    x="Pclass",
    hue="Survived",
    multiple="dodge",
    bins=3,
    shrink=0.8,
    discrete = True,
    ax=ax2
)
ax2.set_xlabel("Ticket Class")
ax2.set_xticks([1,2,3]) 
ax2.set_xticklabels(['1st class', '2nd class', '3rd class'])
ax2.set_ylabel("Number of passengers")
ax2.set_title("Distribution of survivors by ticket class")

sns.histplot(
    data=unprocessed_data[['Survived', 'Sex']].dropna(),
    x="Sex",
    hue="Survived",
    multiple="dodge",
    shrink=0.8,
    discrete = True,
    ax=ax3
)
ax3.set_xlabel("Sex")
ax3.set_ylabel("Number of passengers")
ax3.set_title("Distribution of survivors by sex")
ax3.set_xticks([0,1]) 
ax3.set_xticklabels(['Male', 'Female'])

plt.tight_layout()
plt.show()

In [ ]:
# Let's create a new feature by combining the 'sibsp' (Siblings / Spouses) and 'parch' (Parents / Children) features to create a unique feature of all relatives aboard
unprocessed_data['AllRelatives'] = unprocessed_data[['SibSp', 'Parch']].sum(axis=1)

In [ ]:
# Let's plot the initial features ('sibsp' (Siblings / Spouses) and 'parch' (Parents / Children) features) and the one created ('AllRelatives')
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
sns.histplot(
    data=unprocessed_data[['Survived', 'SibSp']].dropna(),
    x="SibSp",
    hue="Survived",
    multiple="dodge",
    bins=8,
    shrink=0.8,
    discrete = True,
    ax=ax1
)
ax1.set_xlabel("Siblings / Spouses aboard the Titanic")
ax1.set_ylabel("Number of passengers")
ax1.set_title("Distribution of survivors by Siblings / Spouses aboard")

sns.histplot(
    data=unprocessed_data[['Survived', 'Parch']].dropna(),
    x="Parch",
    hue="Survived",
    multiple="dodge",
    bins=3,
    shrink=0.8,
    discrete = True,
    ax=ax2
)
ax2.set_xlabel("Parents / Children aboard the Titanic")
ax2.set_ylabel("Number of passengers")
ax2.set_title("Distribution of survivors by Parents / Children aboard")

sns.histplot(
    data=unprocessed_data[['Survived', 'AllRelatives']].dropna(),
    x="AllRelatives",
    hue="Survived",
    multiple="dodge",
    bins=3,
    shrink=0.8,
    discrete = True,
    ax=ax3
)
ax3.set_xlabel("AllRelatives aboard the Titanic")
ax3.set_ylabel("Number of passengers")
ax3.set_title("Distribution of survivors by any relatives aboard")

plt.tight_layout()
plt.show()

In [ ]:
# Internet hint: a more precise way to fill out the missing ages, would be based on the title of the passengers (Mr, Miss, ect)
# Let's extract all the titles of the passengers
titles = []
for name in unprocessed_data['Name']:
    titles.extend(re.findall(r", (.*?)\.", name))

titles_unique = set(titles)
print('Titles of the passengers aboard the titanic are:',titles_unique)

In [ ]:
# Let's plot the age based on the titles the most present aboard (Mr, Mrs, Miss)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
Mr_Mask = unprocessed_data['Name'].str.contains('Mr.', case=False, na=False)
Mr_Ages = unprocessed_data.loc[Mr_Mask, ['Name', 'Age']].dropna()
sns.histplot(
    data=Mr_Ages,
    x="Age",
    multiple="dodge",
    bins=8,
    shrink=0.8,
    ax=ax1
)
ax1.set_xlabel("Age of all the Mr aboard the Titanic")
ax1.set_ylabel("Number of passengers")
ax1.set_title("Distribution of the ages of Mr aboard")

Mrs_Mask = unprocessed_data['Name'].str.contains('Mrs.', case=False, na=False)
Mrs_Ages = unprocessed_data.loc[Mrs_Mask, ['Name', 'Age']].dropna()
sns.histplot(
    data=Mrs_Ages,
    x="Age",
    multiple="dodge",
    bins=6,
    shrink=0.8,
    ax=ax2
)
ax2.set_xlabel("Age of all the Mrs aboard the Titanic")
ax2.set_ylabel("Number of passengers")
ax2.set_title("Distribution of the ages of Mrs aboard")

Miss_Mask = unprocessed_data['Name'].str.contains('Miss.', case=False, na=False)
Miss_Ages = unprocessed_data.loc[Miss_Mask, ['Name', 'Age']].dropna()
sns.histplot(
    data=Miss_Ages,
    x="Age",
    multiple="dodge",
    bins=8,
    shrink=0.8,
    ax=ax3
)
ax3.set_xlabel("Age of all the Miss aboard the Titanic")
ax3.set_ylabel("Number of passengers")
ax3.set_title("Distribution of the ages of Miss aboard")


plt.tight_layout()
plt.show()

In [ ]:
# As seen of the plots above, the title seem correlated to the age so let's fill the missing age based on the title
# First let's calculate the median of all titles on board
dict_titles_age={}
for t in titles_unique:
    title_Mask = unprocessed_data['Name'].str.contains(str(t+'.'), case=False, na=False)
    title_Ages = unprocessed_data.loc[title_Mask, ['Name', 'Age']]
    title_Ages = title_Ages.iloc[:, 1].median()
    dict_titles_age[str(t)]=title_Ages
    dict_titles_age[str(t+'_nb')]=len(unprocessed_data.loc[title_Mask, ['Name', 'Age']])
print('Title & Median Age of the Passengers aboard the Titanic',dict_titles_age)

# And let's filled the unknown age from our dataset with the median of the passengers based on their titles
for t in titles_unique:
    title_mask = unprocessed_data['Name'].str.contains(str(t + '.'), case=False, na=False)
    missing_age_mask = title_mask & unprocessed_data['Age'].isna()
    unprocessed_data.loc[missing_age_mask, 'Age'] = dict_titles_age[str(t)]

In [ ]:
# Finally let's create a new feature of the passengers titles 
# For that let's use only the titles the most represented aboard (>100 people)
unprocessed_data['Titles'] = unprocessed_data['Name'].str.extract(r", (.*?)\.")
unprocessed_data['Titles'] = unprocessed_data['Titles'].map({'Mrs': 1, 'Miss': 2, 'Mr': 3}).fillna(4)

# 02: Processing and scaling of the features
Now let's process our features and create our training and test sets

In [ ]:
# Let's keep only the non NaN values
unprocessed_data = unprocessed_data[['Survived', 'Pclass', 'Sex', 'Age','AllRelatives','Titles']].dropna()

# Let's define X as the all the features choosen, and Y if they survived
X = unprocessed_data[['Pclass', 'Sex', 'Age', 'AllRelatives','Titles']]
Y = unprocessed_data['Survived']

# Let's define the training set and test set 
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 
    test_size=0.2,   # test size
    random_state=42, # random strategy
    stratify=Y       # to keep the same surviving rate in the training set and the test set
)

print(X_test.shape)
print(X_train.shape)

In [ ]:
# Let's process and scale our inputs 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Calculate and save the scaling parameters of the training set (mean, standard deviation)
X_test_scaled  = scaler.transform(X_test) # Apply the same parameters of the training set to the test set 

# Let's process them for the polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly  = poly.transform(X_test_scaled)

# 03: Definition of the model and training
Let's define the model and analyse the outputs on the training and test set

In [ ]:
# Let's implement the Logistic Regression using our lastly defined polynomial features
titanic_model = LogisticRegression(
    C=0.5,           # Regularization strength: 1/lambda. Lower = stronger regularization
                     # (shrinks weights toward 0, prevents overfitting). Try: 0.001 to 100.
    penalty='l2',    # Type of regularization: 'l2' penalizes w², 'l1' penalizes |w|
                     # l2 (Ridge) keeps all features. l1 (Lasso) can zero out some weights.
    max_iter=10000,  # Max number of iterations the solver is allowed to run.
                     # Increase if you get a ConvergenceWarning.
    solver='lbfgs',  # Optimization algorithm used internally (replaces gradient descent).
                     # 'lbfgs' is the default and works well for small/medium datasets.
                     # Other options: 'saga' (supports l1), 'liblinear' (small datasets).
    random_state=42  # Fixes the random seed for reproducibility.
                     # Same number = same results every run.
)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    titanic_model.fit(X_train_poly, Y_train)
    if w and any(issubclass(x.category, ConvergenceWarning) for x in w):
        print("⚠ Pas convergé")

In [ ]:
# Let's check that C, penalty, and solver are actually the optimized parameters here
param_grid = [
    # Combinaison for l2 and the working associated solvers
    {'C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
     'penalty': ['l2'],
     'solver': ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']},
    
    # Combinaison for solvers working with l1 or l2
    {'C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
     'penalty': ['l1', 'l2'],
     'solver': ['liblinear', 'saga']}
]
grid = GridSearchCV(
    LogisticRegression(penalty='l2', max_iter=10000, solver='lbfgs', random_state=42),
    param_grid,
    cv=5,           # cross-validation on the dataset divided into 5 (4 for train, 1 for test)
    scoring='accuracy'
)
grid.fit(X_train_poly, Y_train)
print("Best C:", grid.best_params_)

# And automatically update the model to the best parameters 
titanic_model = grid.best_estimator_

In [ ]:
# Let's calculate the cost and apply the model on the test set 
y_pred_train  = titanic_model.predict(X_train_poly)
y_pred_val    = titanic_model.predict(X_test_poly)
y_proba_train = titanic_model.predict_proba(X_train_poly)
y_proba_val   = titanic_model.predict_proba(X_test_poly)

cost_train = log_loss(Y_train, y_proba_train)
cost_val   = log_loss(Y_test,  y_proba_val)
gap        = cost_val - cost_train

print(f"Cost train : {cost_train:.4f}")
print(f"Cost val   : {cost_val:.4f}")
print(f"Gap        : {gap:.4f}")
print(f"Accuracy train : {accuracy_score(Y_train, y_pred_train):.4f}")
print(f"Accuracy val   : {accuracy_score(Y_test,  y_pred_val):.4f}")
print(classification_report(Y_test, y_pred_val))

In [ ]:
# Let's run a quick diagnostic overfitting/underfitting
if cost_train > 0.5 and cost_val > 0.5:
    print("→ UNDERFITTING")
elif gap > 0.1:
    print("→ OVERFITTING")
else:
    print("→ OK")

# 04: Predictions on the Titanic survivors
Now lets's apply our model and predict the survivors of the competition set

In [ ]:
# Let's read the input data
predictions_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
predictions_data.head()

# Same creation of features and processing as earlier
predictions_data['Sex'] = predictions_data['Sex'].map({'male': 0, 'female': 1})
predictions_data['AllRelatives'] = predictions_data[['SibSp', 'Parch']].sum(axis=1)

for t in titles_unique:
    title_mask = predictions_data['Name'].str.contains(str(t + '.'), case=False, na=False)
    missing_age_mask = title_mask & predictions_data['Age'].isna()
    predictions_data.loc[missing_age_mask, 'Age'] = dict_titles_age[str(t)]  # Apply the same median age as the training and test sets
    
predictions_data['Titles'] = predictions_data['Name'].str.extract(r", (.*?)\.")
predictions_data['Titles'] = predictions_data['Titles'].map({'Mrs': 1, 'Miss': 2, 'Mr': 3}).fillna(4)

# Let's scale the features 
X_prediction = predictions_data[['Pclass', 'Sex', 'Age', 'AllRelatives','Titles']]
X_predictions_scaled  = scaler.transform(X_prediction)
X_predictions_poly  = poly.transform(X_predictions_scaled)

# Let's apply the model 
predictions = titanic_model.predict(X_predictions_poly) 

In [ ]:
# Let's create the doc to submit to the competition
submission = pd.DataFrame({
    "PassengerId": predictions_data["PassengerId"],
    "Survived": predictions.astype(int)
})

submission.to_csv("submission.csv", index=False)